# Inspeção inicial da base de crédito

Aqui eu confiro o que chegou antes de limpar ou modelar: tamanho, tipos, nulos, duplicatas e distribuição do alvo. Esta leitura descreve a base inteira; as escolhas de imputação e os modelos serão ajustados apenas com o treino. Os CSVs originais são somente leitura.


In [1]:
from pathlib import Path
import hashlib
import json
import pandas as pd

RAIZ = Path.cwd()
if not (RAIZ / "credit_risk_dataset.csv").exists():
    RAIZ = RAIZ.parent
hashes = json.loads((RAIZ / "documentacao/integridade_originais.json").read_text())
for nome, esperado in hashes.items():
    assert hashlib.sha256((RAIZ / nome).read_bytes()).hexdigest() == esperado, nome

dados = pd.read_csv(RAIZ / "credit_risk_dataset.csv")
print(f"Registros: {dados.shape[0]:,}; colunas: {dados.shape[1]}")
print(f"Linhas duplicadas além da primeira ocorrência: {dados.duplicated().sum()}")
display(pd.DataFrame({"tipo": dados.dtypes.astype(str), "nulos": dados.isna().sum()}))

Registros: 32,581; colunas: 12
Linhas duplicadas além da primeira ocorrência: 165


,tipo,nulos
person_age,int64,0
person_income,int64,0
person_home_ownership,str,0
person_emp_length,float64,895
loan_intent,str,0
loan_grade,str,0
loan_amnt,int64,0
loan_int_rate,float64,3116
loan_status,int64,0
loan_percent_income,float64,0


In [2]:
display(dados.describe().T)
classes = dados["loan_status"].value_counts().sort_index()
display(pd.DataFrame({"quantidade": classes, "percentual": (classes / len(dados) * 100).round(2)}))

,count,mean,std,min,25%,50%,75%,max
person_age,32581.0,27.734600,6.348078,20.00,23.00,26.00,30.00,144.00
person_income,32581.0,66074.848470,61983.119168,4000.00,38500.00,55000.00,79200.00,6000000.00
person_emp_length,31686.0,4.789686,4.142630,0.00,2.00,4.00,7.00,123.00
loan_amnt,32581.0,9589.371106,6322.086646,500.00,5000.00,8000.00,12200.00,35000.00
loan_int_rate,29465.0,11.011695,3.240459,5.42,7.90,10.99,13.47,23.22
loan_status,32581.0,0.218164,0.413006,0.00,0.00,0.00,0.00,1.00
loan_percent_income,32581.0,0.170203,0.106782,0.00,0.09,0.15,0.23,0.83
cb_person_cred_hist_length,32581.0,5.804211,4.055001,2.00,3.00,4.00,8.00,30.00


,quantidade,percentual
loan_status,,
0,25473,78.18
1,7108,21.82


### Como ler os números

A tabela de tipos aponta onde há dados ausentes. O resumo estatístico mostra quartis e extremos; a contagem de `loan_status` revela o desbalanceamento. Esses números orientam a investigação, mas ainda não dizem, sozinhos, quais linhas são erros.

## Registros que pedem uma segunda visualização

O filtro abaixo mostra idades acima de 100 anos e tempos de emprego acima de 80. Ele serve para investigar, não para apagar linhas. Depois da inspeção, a regra de limpeza usa idade a partir de 120 anos e transforma em nulo os tempos de emprego impossíveis.


In [3]:
suspeitos = dados.loc[(dados.person_age > 100) | (dados.person_emp_length > 80)]
display(suspeitos)
razao = dados.loan_amnt / dados.person_income
diferenca = (razao - dados.loan_percent_income).abs()
print("Proporção próxima da razão já existente:", round(diferenca.le(0.00501).mean(), 4))
print("Rendas não positivas:", dados.person_income.le(0).sum())

,person_age,person_income,person_home_ownership,person_emp_length,loan_intent,loan_grade,loan_amnt,loan_int_rate,loan_status,loan_percent_income,cb_person_default_on_file,cb_person_cred_hist_length
0,22,59000,RENT,123.0,PERSONAL,D,35000,16.02,1,0.59,Y,3
81,144,250000,RENT,4.0,VENTURE,C,4800,13.57,0,0.02,N,3
183,144,200000,MORTGAGE,4.0,EDUCATION,B,6000,11.86,0,0.03,N,2
210,21,192000,MORTGAGE,123.0,VENTURE,A,20000,6.54,0,0.10,N,4
575,123,80004,RENT,2.0,EDUCATION,B,20400,10.25,0,0.25,N,3
747,123,78000,RENT,7.0,VENTURE,B,20000,NaN,0,0.26,N,4
32297,144,6000000,MORTGAGE,12.0,PERSONAL,C,5000,12.73,0,0.00,N,25


Proporção próxima da razão já existente: 0.9868
Rendas não positivas: 0


## O que esta inspeção indica

A classe inadimplente representa 21,82% da base. Por isso, vou avaliar precisão, recall e F1 da classe 1 além da acurácia. Há nulos em juros e tempo de emprego; a escolha entre média e mediana depende da distribuição observada, e os valores serão aprendidos apenas no treino. A coluna `loan_percent_income` já aproxima a razão empréstimo/renda, então não vou deixar as duas razões como preditoras do KNN.

Na próxima etapa, os gráficos ajudam a conferir essas distribuições. Nenhuma linha foi alterada neste notebook.


In [4]:
for nome, esperado in hashes.items():
    assert hashlib.sha256((RAIZ / nome).read_bytes()).hexdigest() == esperado, nome
print("Integridade dos dois CSVs originais confirmada.")

Integridade dos dois CSVs originais confirmada.
